In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import os
os.makedirs("mp2_outputs", exist_ok=True)


In [4]:
df = pd.read_csv("cdamron3_project_summary.csv", sep=";")
print("\n")
print(df.shape)
print("\n")
print("Columns:\n", df.columns.tolist())
print("\n")
print(df.head())



(25776, 5)


Columns:
 ['project_wocid', 'commit_sha1', 'author', 'time', 'commit_message']


           project_wocid                               commit_sha1  \
0  marianattestad_ribbon  0023779e992f3962eecf62e652ba81fe66da3852   
1  marianattestad_ribbon  00c7ec6d1edf1ee4fef4e87b9e8254271f76fd2e   
2  marianattestad_ribbon  013cfce4dde223bddcc8a599db060b3ee06a8881   
3  marianattestad_ribbon  01ded9b5bd9afe66e5469d7864b8fbe83bb1e66c   
4  marianattestad_ribbon  03ea1c89e7669fd6942131c747023ffa08e7372b   

                                            author        time  \
0  Robert Aboukhalil <robert.aboukhalil@gmail.com>  1584246680   
1      Maria Nattestad <maria.nattestad@gmail.com>  1472156867   
2  Robert Aboukhalil <robert.aboukhalil@gmail.com>  1725660419   
3      Maria Nattestad <maria.nattestad@gmail.com>  1472576123   
4  Robert Aboukhalil <robert.aboukhalil@gmail.com>  1723753639   

                                      commit_message  
0                              

In [5]:
df = df.rename(columns= {
    "project_wocid": "project",
    "commit_sha1": "commit",
    "message": "message",
    "commit_message": "message",
})

# df.columns = ['project','commit','author','time','message']

df = df[["project", "commit", "author", "time", "message"]]
df['Month'] = pd.to_datetime(df['time'], unit='s').dt.strftime('%Y-%m')
df.head()

,project,commit,author,time,message,Month
0,marianattestad_ribbon,0023779e992f3962eecf62e652ba81fe66da3852,Robert Aboukhalil <robert.aboukhalil@gmail.com>,1584246680,Small fix\n,2020-03
1,marianattestad_ribbon,00c7ec6d1edf1ee4fef4e87b9e8254271f76fd2e,Maria Nattestad <maria.nattestad@gmail.com>,1472156867,Merge branch 'master' into web-production\n,2016-08
2,marianattestad_ribbon,013cfce4dde223bddcc8a599db060b3ee06a8881,Robert Aboukhalil <robert.aboukhalil@gmail.com>,1725660419,Download & mount BAIs in one shot instead of 1...,2024-09
3,marianattestad_ribbon,01ded9b5bd9afe66e5469d7864b8fbe83bb1e66c,Maria Nattestad <maria.nattestad@gmail.com>,1472576123,Add variants and features in a singleview head...,2016-08
4,marianattestad_ribbon,03ea1c89e7669fd6942131c747023ffa08e7372b,Robert Aboukhalil <robert.aboukhalil@gmail.com>,1723753639,Fix bug with saveSvgAsPng in splitthreader\n,2024-08


In [6]:
projects = sorted(df["project"].unique())
print(f"\Found {len(projects)} projeccts: {projects}\n")

gap_count = {}

for prj in projects:
    print(f"Loop Start - Current Project {prj}\n")
    df1 = df[df["project"] == prj]

    df2 = df1.groupby("Month").size().reset_index(name="Commits")

    df2["time"] = pd.to_datetime(df2["Month"])
    full_range = pd.DataFrame({"time": pd. date_range(start = df2["time"].min(), end=df2["time"].max(), freq="MS")})

    mc = pd.merge(full_range, df2, on="time", how="left")
    mc["Commits"] = mc["Commits"].fillna(0).astype(int)
    mc["Month"] = mc["time"].dt.strftime("%Y-%m")
    mc = mc[["Month", "Commits"]]

    ts_path = f"mp2_outputs/cdamron3_commits_timeseries_{prj}.csv"
    mc.to_csv(ts_path, sep=";", index=False)

    plt.figure(figsize=(12, 5))
    plt.plot(mc["Month"], mc["Commits"],
             marker="o", linestyle="-", markersize=3)
    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(f"Commit Activity: {prj}")
    plt.grid(True)
    plt.tight_layout()
    ax = plt.gca()

    for i, label in enumerate(ax.xaxis.get_ticklabels()):
       if i % 2 == 1:
            label.set_visible(False)
    plt.xticks(rotation=45, fontsize=7)
    plt.savefig(f"mp2_outputs/cdamron3_timeseries_{prj}.png", dpi=600)
    plt.close()

    commits = mc["Commits"].tolist()
    months = mc["Month"].tolist()

    max_len = 0
    curr_len = 0
    curr_start = None
    best_start = None
    best_end = None

    for m, c in zip(months, commits):
        if c == 0:
            if curr_len == 0:
                curr_start = m
            curr_len += 1
            if curr_len > max_len:
                max_len = curr_len
                best_start = curr_start
                best_end = m
        else:
            curr_len = 0

    num_gaps = 0
    curr = 0
    for c in commits:
        if c == 0:
            curr += 1
        else:
            if curr >= 3:
               num_gaps += 1
            curr = 0
    if curr >= 3:
        num_gaps += 1

    commits_after = 0
    if best_end:
       index = months.index(best_end)
       commits_after = sum(commits[index+1:])

    gap_count[prj] = {
        "LongestGapStart": best_start,
        "LongestGapEnd": best_end,
        "LongestGapLength": max_len,
        "NumCommitsAfterLastGap": commits_after,
        "NumGapsTimeline": num_gaps,
    }

    print(f" Longest Gap: {best_start} -> {best_end} ({max_len} months)\n")
    print(f" gaps >= 3 months: {num_gaps}, commits after gap: {commits_after}\n")

gap_df = pd.DataFrame.from_dict(gap_count, orient="index").reset_index()
gap_df.rename(columns={"index": "Project"}, inplace=True)
gap_df.to_csv(f"mp2_outputs/cdamron3_gap_summary.csv", sep=";", index=False)

print("Done!")

\Found 10 projeccts: ['jhuapl-boss_intern', 'jonescompneurolab_hnn', 'lpwgroup_crank', 'marianattestad_ribbon', 'mirnylab_cooler', 'privefl_bigstatsr', 'rveltz_piecewisedeterministicmarkovprocesses.jl', 'stan-dev_cmdstanpy', 'thelfer_mfrontgenericinterfacesupport', 'xeokit_xeokit-sdk']

Loop Start - Current Project jhuapl-boss_intern

 Longest Gap: 2024-01 -> 2024-05 (5 months)

 gaps >= 3 months: 6, commits after gap: 20

Loop Start - Current Project jonescompneurolab_hnn

 Longest Gap: 2022-09 -> 2023-09 (13 months)

 gaps >= 3 months: 3, commits after gap: 73

Loop Start - Current Project lpwgroup_crank

 Longest Gap: 2022-04 -> 2023-03 (12 months)

 gaps >= 3 months: 6, commits after gap: 20

Loop Start - Current Project marianattestad_ribbon

 Longest Gap: 2017-11 -> 2019-09 (23 months)

 gaps >= 3 months: 7, commits after gap: 302

Loop Start - Current Project mirnylab_cooler

 Longest Gap: 2021-05 -> 2021-10 (6 months)

 gaps >= 3 months: 5, commits after gap: 401

Loop Start - 

In [7]:
# Just to Check Everything:

files = sorted(os.listdir("mp2_outputs"))
print(f"{len(files)} files in mp2_outputs:\n")
for f in files:
    print(" ", f)


22 files in mp2_outputs:

  .ipynb_checkpoints
  cdamron3_commits_timeseries_jhuapl-boss_intern.csv
  cdamron3_commits_timeseries_jonescompneurolab_hnn.csv
  cdamron3_commits_timeseries_lpwgroup_crank.csv
  cdamron3_commits_timeseries_marianattestad_ribbon.csv
  cdamron3_commits_timeseries_mirnylab_cooler.csv
  cdamron3_commits_timeseries_privefl_bigstatsr.csv
  cdamron3_commits_timeseries_rveltz_piecewisedeterministicmarkovprocesses.jl.csv
  cdamron3_commits_timeseries_stan-dev_cmdstanpy.csv
  cdamron3_commits_timeseries_thelfer_mfrontgenericinterfacesupport.csv
  cdamron3_commits_timeseries_xeokit_xeokit-sdk.csv
  cdamron3_gap_summary.csv
  cdamron3_timeseries_jhuapl-boss_intern.png
  cdamron3_timeseries_jonescompneurolab_hnn.png
  cdamron3_timeseries_lpwgroup_crank.png
  cdamron3_timeseries_marianattestad_ribbon.png
  cdamron3_timeseries_mirnylab_cooler.png
  cdamron3_timeseries_privefl_bigstatsr.png
  cdamron3_timeseries_rveltz_piecewisedeterministicmarkovprocesses.jl.png
  cdamron

In [8]:
with open("mp2_outputs/cdamron3_commits_timeseries_marianattestad_ribbon.csv") as f:
    for _ in range(5):
        print(f.readline().rstrip())


Month;Commits
2016-06;4
2016-07;35
2016-08;100
2016-09;34


# Interpretation

...work on this tommorrow...

# Identify the Longest Inactivity Gap

2022-02 to 2023-04

Add all that info to netid_project_stats.csv